# H4 분석 계획 — 쉽게 정리

## 무엇을 확인하려는가
"프로필을 등록 안 한 고객군(2,175명)에게 나간 오퍼는, 등록한 고객보다 남는 돈(순이익)이 적을 것이다" — 이 한 문장을 5단계로 나눠서 확인한다.

## 전체 그림 (읽는 순서)

```
① H4-1  : 진짜 낮은지 본다
② H4-1b : 착시 아닌지 확인 ①  — 오퍼 종류(bogo/discount) 때문은 아닌가?
③ H4-1c : 착시 아닌지 확인 ②  — 무임승차 섞임 때문은 아닌가?
④ H4-2  : 그럼 "왜" 낮은가? → 혹시 미등록 그룹 안에 무임승차 건이 많이 섞여있어서, 그게 그 그룹의 평균을 끌어내린 거 아닐까? 만약 무임승차 건이 정상완료 건보다 금액이 작다면, 무임승차가 많이 섞일수록 전체 평균이 낮아짐
⑤ H4-2b : ④를 한 번 더 보충 확인
```

## 각 단계 상세

### ① H4-1: 정말 낮은가?
- **방법**: 등록 vs 미등록, 완료 한 건당 순이익(profit = 실제 결제금액 − 리워드)을 Mann-Whitney U 검정으로 비교
- **왜 이 검정**: profit은 소수의 초고액 결제 때문에 분포가 심하게 치우쳐 있어서, 평균 기반의 t검정 대신 순위 기반의 Mann-Whitney U를 쓴다

### ② H4-1b: 오퍼 종류 때문에 생긴 착시는 아닌가?
- **왜 이걸 확인하나**: 예전에 다른 가설(H1-2)에서 "bogo와 discount를 합쳐서 보면 착시가 생기고, 나눠 봐야 진짜가 보이는" 경험이 있었다. 그래서 H4-1도 같은 함정이 있는지 미리 점검한다
- **방법**: bogo 안에서만, discount 안에서만 각각 다시 검정

### ③ H4-1c: 무임승차 노이즈 때문에 생긴 착시는 아닌가?
- **왜 이걸 확인하나**: profit에는 "오퍼와 상관없이 일어난 소비(무임승차)"도 섞여 있다. 이 노이즈를 걷어내고 "진짜 오퍼가 유도한 완료"만 남겨도 같은 결과가 나오는지 확인한다
- **방법**: 무임승차 건을 빼고, 남은 것만으로 등록 vs 미등록 재검정

### ④ H4-2: 낮은 이유를 "무임승차가 많아서"라는 설명으로 검증 가능한가?
> 주의: 카이제곱/Mann-Whitney U는 연관성을 보는 검정이라, 이 단계는 인과관계 증명이 아니라 "이 설명이 통계적으로 그럴듯한지" 확인하는 단계다.
- **논리**: "미등록 그룹 안에 (정상완료에 비해 금액이 적은) 무임승차가 더 많이 섞여 있어서 평균이 낮아진 것 아닐까?"라는 가장 그럴듯한 설명을 확인한다.(미등록 고객은 앱에 안 진지하니 습관적인 작은 소비만 하다가 우연히 무임승차했을 것에서 나온 생각임.) 이 설명이 맞으려면 **미등록 그룹의 무임승차 비율이 등록 그룹보다 높아야 한다**
- **방법**: 등록 vs 미등록의 무임승차 비율을 카이제곱 검정으로 비교

### ⑤ H4-2b: (보충) 무임승차가 정말 "금액을 깎는" 요인인지
- **왜 이걸 확인하나**: ④가 성립하려면 "무임승차 건은 원래 금액이 작다"는 것도 사실이어야 한다. 이걸 등록/미등록 그룹 각각 안에서 따로 확인한다
- **방법**: 각 그룹 안에서, 무임승차 건 vs 정상완료 건의 금액을 Mann-Whitney U로 비교

## 무임승차(freeride)란
"이 오퍼가 없었어도 어차피 일어났을 소비"를 뜻한다. 다음 둘 중 하나면 무임승차로 본다.
- 오퍼를 아예 열람하지 않음
- 열람은 했지만, 이미 결제(완료)한 다음에 뒤늦게 열람함 (그 열람이 결제에 영향을 줄 수 없었으므로)

## 유의수준
① ~ ⑤ 모두 α=0.05를 그대로 사용한다. (사전에 이론적 근거가 있는 확증적 검정이라 다중비교 보정을 하지 않는다 — 근거 없이 여러 변수를 찔러보는 탐색적 분석과는 다른 취급)

In [ ]:
"""
H4 가설 검증 — "프로필 미등록 고객군은 순이익 기여도가 낮다"
================================================================

전체 흐름을 한 문장으로 요약하면:
  1) 진짜 낮은지 확인한다 (H4-1)
  2) 착시가 아닌지 두 가지 방법으로 다시 확인한다 (H4-1b, H4-1c)
  3) "왜 낮은지" 설명할 수 있는 후보 가설(무임승차)을 검증한다 (H4-2, H4-2b)
     * 주의: 아래 검정들은 전부 카이제곱/Mann-Whitney U 같은 "연관성" 검정이다.
       교란변수를 통제하거나 실험설계를 한 게 아니므로, 인과관계를 증명하는 건 아니다.
       "원인을 증명했다"가 아니라 "이 설명 가설이 통계적으로 그럴듯한지 확인했다"로 이해할 것.

각 단계마다 "왜 이걸 보는가"를 print로 먼저 설명하고, 그 다음에 결과를 보여준다.
"""

import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.max_columns", None)


def line():
    print("\n" + "=" * 70)


def why(text):
    print(f"\n[왜 이걸 보는가]\n{text}")


def result(text):
    print(f"\n[결과]\n{text}")


def so_what(text):
    print(f"\n[그래서?]\n{text}")


# =========================================================
# 0. 준비
# =========================================================
line()
print("STEP 0. 데이터 준비")
line()

df = pd.read_csv("offer_instance_table.csv")
completed = df[df["is_completed"] == 1].copy()
completed = completed[completed["matched_amount"].notna()]

# 순이익 = 실제 결제금액 - 지급한 리워드
completed["profit"] = completed["matched_amount"] - completed["reward"]


# 무임승차란: "이 오퍼가 없었어도 어차피 일어났을 소비"
#   - 아예 열람 안 함 (is_viewed == 0)
#   - 또는 열람은 했지만, 이미 완료(결제)한 다음에 뒤늦게 열람 (viewed_time > completed_time)
#     -> 이 경우 열람이 완료라는 행동에 영향을 줄 수 없었으므로 무임승차로 취급
def is_freeride(row):
    if row["is_viewed"] == 0:
        return True
    if row["viewed_time"] > row["completed_time"]:
        return True
    return False


completed["freeride"] = completed.apply(is_freeride, axis=1)

print(f"완료 건수: {len(completed):,}건")
print(f"그중 무임승차: {completed['freeride'].sum():,}건 "
      f"({completed['freeride'].mean()*100:.1f}%)")


def mannwhitney(g0, g1, name0, name1):
    """두 그룹의 profit(또는 다른 값)을 비교하는 표준 검정.
    profit은 왜도가 커서(소수의 초고액 결제가 평균을 왜곡) t검정 대신
    순위 기반의 Mann-Whitney U 검정을 쓴다."""
    u, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
    rbc = 1 - (2 * u) / (len(g0) * len(g1))
    print(f"  {name0}: n={len(g0):,}, 중앙값={g0.median():.2f}")
    print(f"  {name1}: n={len(g1):,}, 중앙값={g1.median():.2f}")
    print(f"  p-value={p:.6f}  (0.05보다 작으면 '우연이 아니다')")
    print(f"  효과크기(rank-biserial)={rbc:.4f}  "
          f"(절대값 0.1미만=매우작음, 0.3미만=작음, 0.5미만=중간, 그이상=큼)")
    return p, rbc


def chi2(ct, name):
    """두 그룹의 '비율'을 비교하는 검정 (예: 무임승차 비율)."""
    chi2_stat, p, dof, expected = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2_stat / (n * (min(ct.shape) - 1)))
    print(ct)
    print(f"  p-value={p:.6f}")
    print(f"  효과크기(Cramér's V)={v:.4f}")
    return p, v


# =========================================================
# H4-1. 정말로 미등록 고객군이 순이익이 낮은가?
# =========================================================
line()
print("H4-1. 등록 vs 미등록, 완료당 순이익 비교 (제일 먼저 확인할 것)")
line()

why("""가설 트리에서 처음 세운 질문 그대로다.
프로필을 등록 안 한 고객(2,175명)에게 나간 오퍼가, 등록한 고객에게 나간 오퍼보다
완료 한 건당 남는 돈(순이익)이 적은지 본다.""")

g0 = completed[completed["missing_profile"] == 0]["profit"]
g1 = completed[completed["missing_profile"] == 1]["profit"]
result("")
mannwhitney(g0, g1, "등록", "미등록")

so_what("""등록 중앙값 12.03 vs 미등록 1.98로, 미등록 쪽이 훨씬 낮다.
효과크기도 커서(약 0.63), 우연이 아니라 실질적으로 큰 차이다.
-> 다만 이게 착시가 아닌지 두 가지 방법으로 더 확인해본다 (H4-1b, H4-1c)""")


# =========================================================
# H4-1b. 혹시 오퍼 종류(bogo/discount) 때문에 생긴 착시는 아닐까?
# =========================================================
line()
print("H4-1b. bogo / discount 각각 안에서 따로 재확인")
line()

why("""이전에 다른 가설(H1-2)에서, bogo와 discount를 합쳐서 보면 착시가 생기고
나눠서 봐야 진짜 패턴이 보이는 경우가 있었다.
그래서 H4-1도 혹시 "미등록 고객이 유독 특정 오퍼 종류만 많이 받아서 생긴 차이"는
아닌지, bogo/discount 각각 안에서 따로 확인해본다.""")

for t in ["bogo", "discount"]:
    sub = completed[completed["offer_type"] == t]
    g0 = sub[sub["missing_profile"] == 0]["profit"]
    g1 = sub[sub["missing_profile"] == 1]["profit"]
    print(f"\n--- {t} 안에서 ---")
    mannwhitney(g0, g1, "등록", "미등록")

so_what("""bogo 안에서도, discount 안에서도 똑같이 미등록이 낮다.
-> 특정 오퍼 종류 때문에 생긴 착시가 아니라, 오퍼 종류와 무관한 진짜 차이다.""")


# =========================================================
# H4-1c. 무임승차(오퍼와 무관한 소비) 노이즈를 걷어내도 여전히 낮을까?
# =========================================================
line()
print("H4-1c. 무임승차를 빼고, '진짜 오퍼가 만든 완료'만으로 재확인")
line()

why("""profit 계산에는 무임승차 건(오퍼와 상관없이 일어난 소비)도 섞여 있다.
혹시 이 노이즈 때문에 미등록 그룹이 낮게 보이는 착시는 아닐까?
무임승차를 완전히 빼고, 오퍼가 진짜로 유도한 완료만 남겨서 다시 비교한다.""")

genuine = completed[completed["freeride"] == False]  # 무임승차 제외, 진짜 반응만
g0 = genuine[genuine["missing_profile"] == 0]["profit"]
g1 = genuine[genuine["missing_profile"] == 1]["profit"]
mannwhitney(g0, g1, "등록", "미등록")

so_what("""노이즈를 걷어내도 여전히 미등록이 훨씬 낮다 (효과크기 약 0.61, H4-1과 거의 동일).
-> 무임승차 섞임 때문에 생긴 착시가 아니라, 이것도 진짜 차이임이 다시 확인됐다.

*** 여기까지 정리: H4-1의 "미등록 그룹은 순이익이 낮다"는 결론은
    세 가지 다른 방법으로 확인해도 흔들리지 않는, 신뢰할 수 있는 결과다. ***""")


# =========================================================
# H4-2. 그럼 무임승차 때문일까? -> 설명 가설 검증 (인과증명 아님, 연관성 확인)
# =========================================================
line()
print("H4-2. 왜 낮을까? - 혹시 미등록 그룹에 무임승차가 더 많이 섞여서?")
line()

why("""이제 그럴듯한 설명 하나를 검증해본다. 흔히 생각할 수 있는 설명은 이거다:
"미등록 그룹 안에 '오퍼와 상관없는 소비(무임승차)'가 더 많이 섞여 있어서,
 그게 평균을 깎아내린 게 아닐까?"
이 설명이 맞으려면, 미등록 그룹의 무임승차 '비율'이 등록 그룹보다 높아야 한다.
그래서 비율부터 비교해본다.""")

ct = pd.crosstab(completed["missing_profile"], completed["freeride"])
ct.index = ["등록(0)", "미등록(1)"]
ct.columns = ["정상완료", "무임승차"]
chi2(ct, "무임승차 비율 비교")

so_what("""반대로 나왔다! 미등록 그룹의 무임승차 비율(15.3%)이 오히려
등록 그룹(30.2%)보다 낮다.
-> "무임승차가 더 많이 섞여서 낮다"는 설명은 이 시점에서 이미 기각된다.
   (미등록 그룹은 무임승차가 오히려 적은데도 순이익은 훨씬 낮으니까)""")


# =========================================================
# H4-2b. (보충) 무임승차가 애초에 '금액을 깎는' 요인이 맞는지도 확인
# =========================================================
line()
print("H4-2b. (보충 확인) 무임승차 여부가 실제로 완료 금액에 영향을 주는지")
line()

why("""H4-2에서 이미 결론이 났지만, 혹시 몰라 한 가지 더 확인한다.
"무임승차 건은 원래 금액이 작다"는 게 사실이어야 저 설명(구성비 때문에 평균이 낮다)이
성립할 수 있다. 등록/미등록 각 그룹 안에서, 무임승차 여부에 따라 금액이 다른지 본다.""")

for mp, label in [(0, "등록"), (1, "미등록")]:
    sub = completed[completed["missing_profile"] == mp]
    g0 = sub[sub["freeride"] == False]["profit"]
    g1 = sub[sub["freeride"] == True]["profit"]
    print(f"\n--- {label} 그룹 안에서 ---")
    mannwhitney(g0, g1, "정상완료", "무임승차")

so_what("""등록 그룹 안에서는 무임승차 쪽 금액이 오히려 더 크다(정상적인 소비자일수록
무임승차하기 쉽다는 뜻으로 해석 가능).
미등록 그룹 안에서는 정상완료든 무임승차든 금액 차이가 아예 없다(둘 다 그냥 작다).

*** 최종 결론:
    미등록 그룹의 낮은 순이익은 "무임승차가 많이 섞여서"가 아니라,
    "그 그룹은 무임승차든 정상완료든 상관없이 거래 자체가 원래 작기 때문"이다. ***""")


# =========================================================
# 최종 요약
# =========================================================
line()
print("전체 결론 요약")
line()
print("""
1. 미등록 고객군은 완료당 순이익이 확실히 낮다 (H4-1)
2. 오퍼 종류(H4-1b)나 무임승차 노이즈(H4-1c) 때문에 생긴 착시가 아니다
3. "무임승차가 많아서"라는 설명은 통계적으로 뒷받침되지 않는다 (H4-2, H4-2b)
4. 대신 이 세그먼트는 원래 거래 규모(씀씀이) 자체가 작다는 점과 연관돼 있다
   (※ 이건 연관성 확인이며, 인과관계를 증명한 것은 아니다)

액션 제안:
  - 오퍼 구조를 아무리 손봐도 이 세그먼트는 개선되기 어려움
  - 이 세그먼트 전용으로 더 작은 reward/difficulty 오퍼를 시범 설계하거나
  - 발송 대상에서 축소하는 방향을 검토해야 함
""")

# H4 결과 요약 — 쉽게 정리

## 한 줄 결론
**미등록 고객군은 순이익이 확실히 낮다. 근데 그 이유는 "무임승차가 많아서"가 아니라, 원래 씀씀이(거래 규모) 자체가 작기 때문이다.**

## 단계별 결과

### ① H4-1: 진짜 낮은가? → 네, 확실히 낮다

| | 등록 | 미등록 |
|---|---|---|
| 완료당 순이익 중앙값 | 12.03 | **1.98** |

효과크기(rank-biserial) = 0.63 → **큰 차이**

### ② H4-1b: 오퍼 종류 때문에 생긴 착시인가? → 아니다

| | 등록 중앙값 | 미등록 중앙값 |
|---|---|---|
| bogo 안에서 | 9.57 | 1.70 |
| discount 안에서 | 14.33 | 2.27 |

→ 둘 다에서 똑같이 낮음. 오퍼 종류와 무관한 진짜 차이.

### ③ H4-1c: 무임승차 노이즈 때문에 생긴 착시인가? → 아니다

무임승차를 다 빼고 "진짜 오퍼가 유도한 완료"만 남겨도:

| | 등록 | 미등록 |
|---|---|---|
| 중앙값 | 11.51 | 2.10 |

효과크기 0.61 (①의 0.63과 거의 동일) → 노이즈 문제가 아니라 진짜 차이.

**→ 여기까지: ①의 결론은 3가지 다른 방법으로 확인해도 흔들리지 않는 견고한 결과.**

### ④ H4-2: "무임승차가 많아서"라는 설명으로 낮은 순이익이 설명되는가? → 아니다, 오히려 반대
> (참고: 이 검정은 연관성만 확인하는 것이며, 인과관계를 증명하는 건 아니다)

| | 등록 | 미등록 |
|---|---|---|
| 완료 중 무임승차 비율 | 30.2% | **15.3%** |

→ 미등록 그룹은 무임승차가 오히려 **더 적다.** "무임승차가 많이 섞여서 낮다"는 설명은 이 자체로 이미 성립 불가.

### ⑤ H4-2b: (보충) 무임승차가 정말 금액을 깎는가?

| | 정상완료 중앙값 | 무임승차 중앙값 | 차이 유의한가 |
|---|---|---|---|
| 등록 그룹 안에서 | 11.51 | 13.25 | 유의함 (무임승차가 오히려 더 큼) |
| 미등록 그룹 안에서 | 2.10 | 1.52 | **유의하지 않음** (거의 차이 없음) |

→ 등록 그룹에서조차 "무임승차 = 작은 금액"이 아니었고(오히려 큼), 미등록 그룹에서는 무임승차든 아니든 그냥 둘 다 작다.

## 그래서 최종 결론

미등록 고객군의 낮은 순이익은:
- 오퍼 종류 때문도 아니고 (②)
- 무임승차가 섞여서 생긴 착시도 아니고 (③)
- 무임승차 비율이 높아서도 아니다 (④⑤)

**→ 이 세그먼트는 무임승차든 정상 반응이든 상관없이, 원래 거래 규모(씀씀이) 자체가 작다는 것과 연관돼 있다.** (※ 이는 통계적 연관성이며, 인과관계를 증명한 것은 아니다) 오퍼 구조를 아무리 개선해도 이 세그먼트의 수익성은 근본적으로 개선되기 어려울 것으로 보인다.

## 액션 제안

1. **오퍼 구조 개선(반응률을 높이자)이라는 일반 처방을 이 세그먼트에는 적용하지 않는다** — 무임승차 비율이 낮은 세그먼트라, 그 처방이 통계적으로 뒷받침되지 않기 때문
2. **이 세그먼트 전용으로 더 작은 reward·difficulty 오퍼를 시범 설계**하거나
3. **발송 대상에서 축소를 검토** — 다만 바로 전면 배제하기보다, 소규모 A/B 테스트로 먼저 확인 권장 (미등록 그룹 표본이 1,101건으로 상대적으로 작기 때문)